In [1]:
!pip install torch torchvision numpy matplotlib tqdm

Define Inital classes

In [2]:
CLASSES = [
    "apple",
    "cloud",
    "star",
    "tree",
    "cup"
]

Download Dataset

In [3]:
import urllib.request
import os

BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap/"

os.makedirs("data", exist_ok=True)

for cls in CLASSES:
  filename = f"{cls}.npy"
  url = BASE_URL + filename.replace(" ", "%20")
  print(f"Downloading {url}")
  urllib.request.urlretrieve(url, os.path.join("data", filename))

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset

class QuickDrawDataset(Dataset):

  def __init__(self, images, labels):
    self.images = images
    self.labels = labels

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    image = self.images[idx]
    label = self.labels[idx]

    image = image.reshape(28, 28)
    image = image.astype(np.float32) / 255.0
    image = torch.tensor(image).unsqueeze(0)

    return image, label

Load Dataset

In [5]:
def load_data(classes, samples_per_class=5000):
  images = []
  labels = []

  for idx, cls in enumerate(classes):
    data = np.load(f"data/{cls}.npy")
    data = data[:samples_per_class]
    images.append(data)
    labels.extend([idx] * len(data))

  images = np.concatenate(images)

  return images, labels

Train/Test Split

In [6]:
from sklearn.model_selection import train_test_split

images, labels = load_data(CLASSES)

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=0.2,
    random_state=42
)

Create DataLoaders

In [7]:
from torch.utils.data import DataLoader

train_dataset = QuickDrawDataset(X_train, y_train)
test_dataset = QuickDrawDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

Build CNN model

In [8]:
import torch.nn as nn

class QuickDrawCNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()

    self.features = nn.Sequential(
        nn.Conv2d(1, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 128),
        nn.ReLU(),

        nn.Dropout(0.3),
        nn.Linear(128, num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x


Training Setup

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = QuickDrawCNN(
    num_classes=len(CLASSES)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

Training Loop

In [11]:
from tqdm import tqdm

EPOCHS = 10

for epoch in range(EPOCHS):
  model.train()
  running_loss = 0

  for images, labels in tqdm(train_loader):
    images = images.to(device)
    labels = labels.to(device) # Fixed: Changed from images.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item()

  print(
      f"Epoch {epoch + 1}/{EPOCHS} - Loss: {running_loss / len(train_loader)}"
  )

100%|██████████| 313/313 [00:00<00:00, 518.91it/s]


Epoch 1/10 - Loss: 0.3811183859603093


100%|██████████| 313/313 [00:00<00:00, 837.00it/s]


Epoch 2/10 - Loss: 0.16859365921741287


100%|██████████| 313/313 [00:00<00:00, 830.37it/s]


...

Epoch 8/10 - Loss: 0.061152830847149224


100%|██████████| 313/313 [00:00<00:00, 894.32it/s]


Epoch 9/10 - Loss: 0.0551332466102714


100%|██████████| 313/313 [00:00<00:00, 894.84it/s]

Epoch 10/10 - Loss: 0.04948175083935118


Evaluation

In [14]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
  for images, labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)

    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 96.76%


Save Model

In [15]:
import torch
import json

torch.save(
    model.state_dict(),
    "best_model.pth"
)

with open("classes.json", "w") as f:
  json.dump(CLASSES, f)